In [1]:
%load_ext autoreload 
%autoreload 2
%reload_ext autoreload

# 1) At the very top of your script or notebook, before you import tensorflow:
import os
# suppress C++ INFO and WARNING logs (0 = all, 1 = filter INFO, 2 = filter WARNING, 3 = filter ERROR)
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
    
import shap
from skimage.segmentation import slic
import tensorflow as tf
# force TF to retain *every* intermediate tensor (including control‐flow outputs)
#Required for running gradientexplainer
tf.compat.v1.experimental.output_all_intermediates(True)

import numpy as np
from tensorflow.keras.models import load_model
from function import losses
from tensorflow.keras.layers import SpatialDropout2D, Dropout, Lambda
import time
import keras_cv
from tqdm import tqdm
import random

IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html


Using TensorFlow backend


## Code to investigate the impact of the number of background samples to estimate robustness

In [2]:

# 2) Suppress Python warnings (including deprecation warnings)
import warnings
warnings.filterwarnings('ignore', category=DeprecationWarning)
warnings.filterwarnings('ignore', category=FutureWarning)

# 3) Import and quiet TensorFlow’s Python logger
import logging

# this silences most tf.get_logger messages
tf.get_logger().setLevel(logging.ERROR)
# and this silences lower-level logs from the 'tensorflow' namespace
logging.getLogger('tensorflow').setLevel(logging.ERROR)


In [3]:
region_name = "CONUS"
experiment = "EX29"
week_lead = 2
ref_source = "GEFSv12"

checkpoint = f'checkpoints/{region_name}/Wk{week_lead}/Wk{week_lead}_{experiment}_regular_RZSM' if \
ref_source == 'GEFSv12' else f'checkpoints/{region_name}/Wk{week_lead}/Wk{week_lead}_{experiment}_ECMWF_regular_RZSM'
print(checkpoint)

checkpoints/CONUS/Wk2/Wk2_EX29_regular_RZSM


In [4]:
def select_n_channels(week_lead, ref_source):
    # return just the channel count. These are known beforehand because of how the models were setup
    if week_lead == 1 and ref_source == 'GEFSv12':
        return 13
    if week_lead == 2 and ref_source == 'GEFSv12':
        return 14
    if week_lead == 3 and ref_source == 'GEFSv12':
        return 5
    if week_lead == 4 and ref_source == 'GEFSv12':
        return 6
    if week_lead == 5 and ref_source == 'GEFSv12':
        return 7
    raise ValueError(f"Unsupported lead={week_lead}, source={ref_source}")

def return_data_dir(week_lead, ref_source,region_name):
    return (f'Data/model_npy_input_data/{region_name}/Wk{week_lead}_EX_input_data' if ref_source == 'GEFSv12' else  f'Data/{model_npy_input_data}/{region_name}/Wk{week_lead}_ECMWF_EX_input_data')


# 1a) Define a function that maps dropout → identity
def remove_dropout(layer):
    """
    Replace any SpatialDropout2D or Dropout layer with a no-op identity layer.
    All other layers are returned unchanged.
    """
    if isinstance(layer, (SpatialDropout2D, Dropout, keras_cv.layers.SqueezeAndExcite2D)):
        return Lambda(lambda x: x)
    return layer

# Load training and testing data

In [5]:
X_train = np.load(f'{return_data_dir(week_lead, ref_source,region_name)}/{experiment}_RZSM_training_input.npy')
X_test = np.load(f'{return_data_dir(week_lead, ref_source,region_name)}/{experiment}_RZSM_testing_input.npy')

print(X_train[0,:,:,0])

[[0.42408124 0.4734531  0.4485158  ... 0.43075514 0.42779055 0.42665896]
 [0.4341465  0.44612762 0.4472022  ... 0.         0.4281042  0.        ]
 [0.4442203  0.44606608 0.4603065  ... 0.42894194 0.4409061  0.45954335]
 ...
 [0.         0.         0.         ... 0.         0.         0.        ]
 [0.         0.         0.         ... 0.         0.         0.        ]
 [0.         0.         0.         ... 0.         0.         0.        ]]


# Load model and channel list information

In [6]:
#Add custom loss function
model = load_model(
    checkpoint,
    custom_objects={"crps2d_tf": losses.crps2d_tf},
    compile=False        # you don’t need to recompile if you’re just doing inference/SHAP
)

#feature channel names (predictors)
with open(f"channel_list_information/Wk{week_lead}/{experiment}_RZSM_channel_list.txt") as f:
    names = [line.strip() for line in f]
names = [s.split()[-1] for s in names]
print(names)

['RZSM_obs_lag-1', 'RZSM_obs_lag-7', 'RZSM_obs_lag-14', 'pwat_obs_lag-1', 'spfh_obs_lag-1', 'tmax_obs_lag-1', 'diff_temp_obs_lag-1', 'z200_obs_lag-1', 'RZSM_prediction_lead1', 'pwat_ref_lead2', 'spfh_ref_lead2', 'tmax_ref_lead2', 'diff_temp_ref_lead2', 'z200_ref_lead2']


# Load model weights and prune layers which are not compatible with Gradient Explainer

In [7]:
# 1) Save the original weights
print("▶ Saving original model weights…")
model.save_weights("/tmp/original_weights.h5")
print("✅ Weights saved to /tmp/original_weights.h5\n")

# 2) Clone & prune the model
def prune_layer(layer):
    # replace Dropout, SpatialDropout2D, and SqueezeAndExcite2D with identity
    if isinstance(layer, (Dropout, SpatialDropout2D, keras_cv.layers.SqueezeAndExcite2D)):
        return Lambda(lambda x: x, name=f"{layer.name}_pruned")
    return layer

print("▶ Cloning and pruning model…")
model_no_do = tf.keras.models.clone_model(
    model,
    clone_function=prune_layer
)
print("✅ Clone+prune complete.\n")

# 3) Load weights by name, skipping mismatches
print("▶ Loading matching weights into pruned model (by_name=True, skip_mismatch=True)…")
model_no_do.load_weights(
    "/tmp/original_weights.h5",
    by_name=True,
    skip_mismatch=True
)
print("✅ Weights loaded.\n")

# 4) Rebuild scalar-output model
print("▶ Building scalar-output model…")
final_head   = model_no_do.output[2]                        # 3rd head
scalar       = tf.reduce_mean(final_head, axis=[1,2,3])     # mean over H, W
scalar_model = tf.keras.Model(inputs=model_no_do.input, outputs=scalar)
print("✅ Scalar model built.")
print("   Inputs:",  scalar_model.input.shape)
print("   Output:",  scalar_model.output.shape)

▶ Saving original model weights…
✅ Weights saved to /tmp/original_weights.h5

▶ Cloning and pruning model…
✅ Clone+prune complete.

▶ Loading matching weights into pruned model (by_name=True, skip_mismatch=True)…
✅ Weights loaded.

▶ Building scalar-output model…
✅ Scalar model built.
   Inputs: (None, 48, 96, 14)
   Output: (None,)


# Quick test on a single sample

In [8]:
import time

def diagnostic_shap_test(
    scalar_model,
    background_data,
    X_data,
    sample_index=0,
    nsamples=1
):
    """
    Diagnostic SHAP test that:
      1) Prints shapes of inputs, background, and model I/O
      2) Computes and reports raw gradient min/max (or notes if None)
      3) Runs a tiny SHAP GradientExplainer and times it
      4) Prints SHAP summary stats

    Parameters
    ----------
    scalar_model : tf.keras.Model
        Maps (None, H, W, C) → (None,) scalar per sample.
    background_data : np.ndarray
        Background for SHAP, shape (N_bg, H, W, C).
    X_data : np.ndarray
        Data to explain, shape (N, H, W, C).
    sample_index : int
        Which sample in X_data to test.
    nsamples : int
        Number of nsamples to request from SHAP (set to 1 for a single pass).

    Returns
    -------
    dt : float
        Seconds taken by the SHAP call.
    shap_vals : np.ndarray
        The raw SHAP array for that single sample.
    """

    # 1) Shapes
    xb = X_data[sample_index:sample_index+1]
    bg = background_data[:1]
    print(f">>> Testing sample #{sample_index}: xb.shape = {xb.shape}")
    print(f">>> Using 1 background sample: bg.shape = {bg.shape}")
    print(f">>> scalar_model.input shape: {tuple(scalar_model.input.shape)}")
    print(f">>> scalar_model.output shape: {tuple(scalar_model.output.shape)}\n")

    # 2) Raw gradient check
    print(">>> Computing raw gradients with GradientTape …")
    x_tf = tf.convert_to_tensor(xb, dtype=tf.float32)
    with tf.GradientTape() as tape:
        tape.watch(x_tf)
        y = scalar_model(x_tf)  # shape [1]
    grads = tape.gradient(y, x_tf)
    if grads is None:
        print("    → Gradient is None: no gradient flow!")
    else:
        min_g = tf.reduce_min(grads).numpy()
        max_g = tf.reduce_max(grads).numpy()
        print(f"    → raw gradient min = {min_g:.3e}, max = {max_g:.3e}")
    print()

    # 3) SHAP call
    print(f">>> Building GradientExplainer(nsamples={nsamples}) …")
    expl = shap.GradientExplainer(scalar_model, bg)
    print(">>> Running explainer.shap_values(xb) …")
    t0 = time.time()
    shap_vals = expl.shap_values(xb, nsamples=nsamples)
    dt = time.time() - t0
    print(f"    → SHAP call took {dt:.2f}s")

    # 4) SHAP stats
    flat = np.array(shap_vals).reshape(-1)
    unique_vals = np.unique(np.round(flat, 6))
    mean_abs    = np.mean(np.abs(flat))
    print(f"    → unique SHAP (rounded 1e-6): {unique_vals[:5]} … {unique_vals[-5:]}")
    print(f"    → mean |SHAP| = {mean_abs:.3e}\n")

    return dt, shap_vals


# 4) Prepare a *background* dataset from TRAINING data
bg_idx     = np.random.choice(X_train.shape[0], size=5, replace=False)
background = X_train[bg_idx, : ,: , :select_n_channels(week_lead, ref_source)]   # shape (size, 48, 96, 14)

# Example usage:
dt, sv = diagnostic_shap_test(scalar_model, background, X_test, sample_index=0, nsamples=1)


>>> Testing sample #0: xb.shape = (1, 48, 96, 14)
>>> Using 1 background sample: bg.shape = (1, 48, 96, 14)
>>> scalar_model.input shape: (None, 48, 96, 14)
>>> scalar_model.output shape: (None,)

>>> Computing raw gradients with GradientTape …
    → raw gradient min = -7.541e-03, max = 2.387e-02

>>> Building GradientExplainer(nsamples=1) …
>>> Running explainer.shap_values(xb) …


`tf.keras.backend.set_learning_phase` is deprecated and will be removed after 2020-10-11. To update it, simply pass a True/False value to the `training` argument of the `__call__` method of your layer or model.


    → SHAP call took 3.29s
    → unique SHAP (rounded 1e-6): [-0.000718 -0.000524 -0.000438 -0.000435 -0.000313] … [0.000319 0.000336 0.000429 0.000448 0.000454]
    → mean |SHAP| = 2.588e-06



# Gradient Explainer

## Runs a different number of background samples to estimate how these impact the variance and shap values of the outputs

In [ ]:
import os
import numpy as np
import shap
from tqdm import tqdm
from datetime import datetime

# 0) Pick a global seed
SEED = 225

# 1) Seed Python, NumPy, and TF
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)


# PARAMETERS
Ks = [10, 25, 50, 100, 250, 500]             # different background‐sample sizes to test
Ks = [500]
batch_size = 10
C = select_n_channels(week_lead, ref_source)
N_train = X_train.shape[0]
N_test  = X_test.shape[0]

# Directory to hold all runs
base_dir = f"Data/shap_tests/Wk{week_lead}"
os.makedirs(base_dir, exist_ok=True)

for K in Ks:
    # 1) Sample K backgrounds
    bg_idx = np.random.choice(N_train, size=K, replace=False)
    background = X_train[bg_idx, : ,: , :C]        # (K,48,96,C)

    # 2) Build Explainer
    explainer = shap.GradientExplainer(scalar_model, background)

    # 3) Compute SHAP values on the test set in batches
    shap_list = []
    print(f"\n▶ Running SHAP with K={K} background samples …")
    for start in tqdm(range(0, N_test, batch_size), desc=f"batches (K={K})"):
        end   = min(start + batch_size, N_test)
        batch = X_test[start:end, : ,: , :C]
        sv    = explainer.shap_values(batch, nsamples=background.shape[0])
        shap_list.append(sv)
    shap_values = np.concatenate(shap_list, axis=0)  # shape (N_test,48,96,C)

    # 4) Compute per-channel statistics
    flat = shap_values.reshape(-1, C)               # (N_test*48*96, C)
    mean_abs = np.mean(np.abs(flat), axis=0)        # (C,)
    var_shap = np.var(flat, axis=0)                 # (C,)

    # 5) Save results in a unique run directory
    run_name  = f"K{K}_background_gradient_explainer"
    run_dir   = os.path.join(base_dir, run_name)
    os.makedirs(run_dir, exist_ok=True)

    np.save(os.path.join(run_dir, "bg_idx.npy"),    bg_idx)
    np.save(os.path.join(run_dir, "shap_values.npy"),shap_values)
    np.save(os.path.join(run_dir, "mean_abs.npy"),   mean_abs)
    np.save(os.path.join(run_dir, "var.npy"),        var_shap)

    print(f"✔ Run K={K} saved to '{run_dir}'")

stop


▶ Running SHAP with K=500 background samples …


batches (K=500):  99%|█████████▉| 114/115 [41:23<00:21, 21.56s/it]